# LEER ALUMNOS

### LIBRERIAS

In [23]:
import pandas as pd

In [24]:
pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [25]:
import sys
!{sys.executable} -m pip install openpyxl

!{sys.executable} -m pip install deltalake
!{sys.executable} -m pip install pyarrow

In [26]:
# Librerias necesarias para el almacenamiento
from deltalake import write_deltalake, DeltaTable
import pyarrow as pa

### FUNCIONES

In [27]:
# Funciones para la ingesta de datos
def get_full_data(base_url, endpoint, data_field=None, params=None, headers=None):
    """
    Realiza una solicitud GET a una API para obtener datos.

    Parámetros:
    base_url (str): La URL base de la API.
    endpoint (str): El endpoint de la API al que se realizará la solicitud.
    params (dict): Parámetros de consulta para enviar con la solicitud.
    data_field (str): El nombre del campo en el JSON que contiene los datos.
    headers (dict): Encabezados para enviar con la solicitud.

    Retorna:
    dict: Los datos obtenidos de la API en formato JSON.
    """
    if params==None:
        params={}

    endpoint_url = f"{base_url}/{endpoint}"
    # Arrays usdados para almacenar los resultados de todas las paginas que devuelva la API
    results=[]
    all_results=[]
    pagina=1
    try:
        while True:
            # En cada iteracion se recorre una pagina con 1000 registros, se los almacena y se pasa a la siguiente pagina.
            params_page={
                "page": pagina,
                "limit": 1000,
                "sort": "period.start", 
                "order": "asc"  
            }
            params = params | params_page
            response = requests.get(endpoint_url, params=params, headers=headers)
            response.raise_for_status()  
            try:
                data = response.json()
                if data_field:
                    results= data.get(data_field,[])
                    all_results.extend(results)
            except:
                print("El formato de respuesta no es el esperado")
                return None
            if len(results) < params["limit"]:
                break
            pagina+=1
    
        return all_results
        # Verificar si los datos están en formato JSON.

    except requests.exceptions.RequestException as e:
        # Capturar cualquier error de solicitud, como errores HTTP.
        print(f"La petición ha fallado. Código de error : {e}")
        return None

def build_table(json_data):
    """
    Construye un DataFrame de pandas a partir de datos en formato JSON.

    Parámetros:
    json_data (dict): Los datos en formato JSON obtenidos de una API.

    Retorna:
    DataFrame: Un DataFrame de pandas que contiene los datos.
    """
    try:
        df = pd.json_normalize(json_data, sep='.', record_path=None)
        return df
    except:
        print("Los datos no están en el formato esperado")
        return None
    

In [28]:
# Funciones para el almacenamiento de datos
def save_data_as_delta(df, path, mode="overwrite", partition_cols=None):
    """
    Guarda un dataframe en formato Delta Lake en la ruta especificada.
    A su vez, es capaz de particionar el dataframe por una o varias columnas.
    Por defecto, el modo de guardado es "overwrite".

    Args:
      df (pd.DataFrame): El dataframe a guardar.
      path (str): La ruta donde se guardará el dataframe en formato Delta Lake.
      mode (str): El modo de guardado. Son los modos que soporta la libreria
      deltalake: "overwrite", "append", "error", "ignore".
      partition_cols (list or str): La/s columna/s por las que se particionará el
      dataframe. Si no se especifica, no se particionará.
    """
    try:
       write_deltalake(
        path, df, mode=mode, partition_by=partition_cols
        )
       print("Datos almacenados en el Datalke en:", path)
    except Exception as e:
        print("Error al cargar datos en Deltake: ", e)

def save_new_data_as_delta(new_data, data_path, predicate, partition_cols=None):
    """
    Guarda solo nuevos datos en formato Delta Lake usando la operación MERGE,
    comparando los datos ya cargados con los datos que se desean almacenar
    asegurando que no se guarden registros duplicados.

    Args:
      new_data (pd.DataFrame): Los datos que se desean guardar.
      data_path (str): La ruta donde se guardará el dataframe en formato Delta Lake.
      predicate (str): La condición de predicado para la operación MERGE.
    """

    try:
      dt = DeltaTable(data_path)
      new_data_pa = pa.Table.from_pandas(new_data)
      # Se insertan en target, datos de source que no existen en target
      dt.merge(
          source=new_data_pa,
          source_alias="source",
          target_alias="target",
          predicate=predicate
      ) \
      .when_not_matched_insert_all() \
      .execute()
    # Si no existe la tabla Delta Lake, se guarda como nueva
    except:
      save_data_as_delta(new_data, data_path, partition_cols=partition_cols)

## BRONZE

### INGESTA

In [29]:
# 1. Leer el Excel
df_students = pd.read_excel("Dataset\Alumnos_Libres_sql.xlsx")

df_students


,Año,especialid,esp,plan,materia,nombre,Total Inscriptos,Nuevos Inscriptos,Nuevos Inscriptos Libres,Recursantes Libres,Nuevos Inscriptos Regulares,Nuevos Inscritpos Promocionados,Recursantes,Recursantes Regulares,Recursantes Promocionados
0,2025,5,Ingeniería en Sistemas de Información ...,2008,101,Álgebra y Geometría Analítica ...,87,5,5,72,0,0,82,8,2
1,2025,5,Ingeniería en Sistemas de Información ...,2008,102,Análisis Matemático I ...,137,13,11,120,1,1,124,3,1
2,2025,5,Ingeniería en Sistemas de Información ...,2008,110,Ingeniería y Sociedad ...,22,2,2,17,0,0,20,1,2
3,2025,5,Ingeniería en Sistemas de Información ...,2008,111,Inglés I ...,27,8,5,13,3,0,19,4,2
4,2025,5,Ingeniería en Sistemas de Información ...,2008,115,Sistemas de Representación ...,43,8,8,33,0,0,35,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
363,2025,31,Ingeniería Civil ...,2023,306,Cálculo Avanzado ...,7,7,0,0,4,3,0,0,0
364,2025,31,Ingeniería Civil ...,2023,307,Instalaciones Eléctricas y Acústicas ...,9,9,3,0,3,3,0,0,0
365,2025,31,Ingeniería Civil ...,2023,308,Instalaciones Termomecánicas ...,8,8,1,0,7,0,0,0,0
366,2025,31,Ingeniería Civil ...,2023,309,Economía ...,42,42,31,0,8,3,0,0,0


### ALMACENAMIENTO

In [30]:
print(df_students.dtypes)

Año                                 int64
especialid                          int64
esp                                object
plan                                int64
materia                             int64
nombre                             object
Total Inscriptos                    int64
Nuevos Inscriptos                   int64
Nuevos Inscriptos Libres            int64
Recursantes Libres                  int64
Nuevos Inscriptos Regulares         int64
Nuevos Inscritpos Promocionados     int64
Recursantes                         int64
Recursantes Regulares               int64
Recursantes Promocionados           int64
dtype: object


In [31]:
# Nos aseguramos de que ninguna columna tenga nulos y definimos los tipos correctos para almacenarlos
df_students["Año"] = df_students["Año"].fillna(0).astype(int)
df_students["especialid"] = df_students["especialid"].fillna(0).astype(int)
df_students["esp"] = df_students["esp"].fillna("").astype(str)
df_students["plan"] = df_students["plan"].fillna(0).astype(int)
df_students["materia"] = df_students["materia"].fillna(0).astype(int)
df_students["nombre"] = df_students["nombre"].fillna("").astype(str)
df_students["Total Inscriptos"] = df_students["Total Inscriptos"].fillna(0).astype(int)
df_students["Nuevos Inscriptos"] = df_students["Nuevos Inscriptos"].fillna(0).astype(int)
df_students["Nuevos Inscriptos Libres"] = df_students["Nuevos Inscriptos Libres"].fillna(0).astype(int)
df_students["Recursantes Libres"] = df_students["Recursantes Libres"].fillna(0).astype(int)
df_students["Nuevos Inscriptos Regulares"] = df_students["Nuevos Inscriptos Regulares"].fillna(0).astype(int)
df_students["Nuevos Inscritpos Promocionados"] = df_students["Nuevos Inscritpos Promocionados"].fillna(0).astype(int)
df_students["Recursantes"] = df_students["Recursantes"].fillna(0).astype(int)
df_students["Recursantes Regulares"] = df_students["Recursantes Regulares"].fillna(0).astype(int)
df_students["Recursantes Promocionados"] = df_students["Recursantes Promocionados"].fillna(0).astype(int)
df_students

,Año,especialid,esp,plan,materia,nombre,Total Inscriptos,Nuevos Inscriptos,Nuevos Inscriptos Libres,Recursantes Libres,Nuevos Inscriptos Regulares,Nuevos Inscritpos Promocionados,Recursantes,Recursantes Regulares,Recursantes Promocionados
0,2025,5,Ingeniería en Sistemas de Información ...,2008,101,Álgebra y Geometría Analítica ...,87,5,5,72,0,0,82,8,2
1,2025,5,Ingeniería en Sistemas de Información ...,2008,102,Análisis Matemático I ...,137,13,11,120,1,1,124,3,1
2,2025,5,Ingeniería en Sistemas de Información ...,2008,110,Ingeniería y Sociedad ...,22,2,2,17,0,0,20,1,2
3,2025,5,Ingeniería en Sistemas de Información ...,2008,111,Inglés I ...,27,8,5,13,3,0,19,4,2
4,2025,5,Ingeniería en Sistemas de Información ...,2008,115,Sistemas de Representación ...,43,8,8,33,0,0,35,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
363,2025,31,Ingeniería Civil ...,2023,306,Cálculo Avanzado ...,7,7,0,0,4,3,0,0,0
364,2025,31,Ingeniería Civil ...,2023,307,Instalaciones Eléctricas y Acústicas ...,9,9,3,0,3,3,0,0,0
365,2025,31,Ingeniería Civil ...,2023,308,Instalaciones Termomecánicas ...,8,8,1,0,7,0,0,0,0
366,2025,31,Ingeniería Civil ...,2023,309,Economía ...,42,42,31,0,8,3,0,0,0


In [32]:
# Filtramos los alumnos del plan 2008
df_plan_2008 = df_students[df_students["plan"] == 2008].copy()

# Filtramos los alumnos del plan 2023
df_plan_2023 = df_students[df_students["plan"] == 2023].copy()

# Filtramos los alumnos del plan 94
df_plan_94 = df_students[df_students["plan"] == 94].copy()

# Filtramos los alumnos del plan 95
df_plan_95 = df_students[df_students["plan"] == 95].copy()


# Vemos uno de los plnes
df_plan_2023

,Año,especialid,esp,plan,materia,nombre,Total Inscriptos,Nuevos Inscriptos,Nuevos Inscriptos Libres,Recursantes Libres,Nuevos Inscriptos Regulares,Nuevos Inscritpos Promocionados,Recursantes,Recursantes Regulares,Recursantes Promocionados
60,2025,5,Ingeniería en Sistemas de Información ...,2023,101,Análisis Matemático I ...,1100,697,553,340,104,40,403,51,12
61,2025,5,Ingeniería en Sistemas de Información ...,2023,102,Álgebra y Geometría Analítica ...,927,633,418,228,152,63,294,57,9
62,2025,5,Ingeniería en Sistemas de Información ...,2023,103,Física I ...,891,516,335,245,138,43,375,116,14
63,2025,5,Ingeniería en Sistemas de Información ...,2023,104,Inglés I ...,467,424,143,18,79,202,43,10,15
64,2025,5,Ingeniería en Sistemas de Información ...,2023,105,Lógica y Estructuras Discretas ...,982,533,288,285,180,65,449,126,38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
363,2025,31,Ingeniería Civil ...,2023,306,Cálculo Avanzado ...,7,7,0,0,4,3,0,0,0
364,2025,31,Ingeniería Civil ...,2023,307,Instalaciones Eléctricas y Acústicas ...,9,9,3,0,3,3,0,0,0
365,2025,31,Ingeniería Civil ...,2023,308,Instalaciones Termomecánicas ...,8,8,1,0,7,0,0,0,0
366,2025,31,Ingeniería Civil ...,2023,309,Economía ...,42,42,31,0,8,3,0,0,0


In [33]:
# Almacenamos los datos en un Deltalake
path_2008="Data/bronze/2008"
path_2023="Data/bronze/2023"
path_1994="Data/bronze/1994"
path_1995="Data/bronze/1995"

save_data_as_delta(df_plan_2008, path_2008, mode="overwrite")
save_data_as_delta(df_plan_2023, path_2023, mode="overwrite")

save_data_as_delta(df_plan_94, path_1994, mode="overwrite")
save_data_as_delta(df_plan_95, path_1995, mode="overwrite")

Datos almacenados en el Datalke en: Data/bronze/2008
Datos almacenados en el Datalke en: Data/bronze/2023
Datos almacenados en el Datalke en: Data/bronze/1994
Datos almacenados en el Datalke en: Data/bronze/1995


In [34]:
# Visualizacion de todos los datos almacenados
dsc_data_2008="Data/bronze/2008"
df_students_2008_stored = DeltaTable(dsc_data_2008).to_pandas() 
df_students_2008_stored[:3]

,Año,especialid,esp,plan,materia,nombre,Total Inscriptos,Nuevos Inscriptos,Nuevos Inscriptos Libres,Recursantes Libres,Nuevos Inscriptos Regulares,Nuevos Inscritpos Promocionados,Recursantes,Recursantes Regulares,Recursantes Promocionados,__index_level_0__
0,2025,5,Ingeniería en Sistemas de Información ...,2008,101,Álgebra y Geometría Analítica ...,87,5,5,72,0,0,82,8,2,0
1,2025,5,Ingeniería en Sistemas de Información ...,2008,102,Análisis Matemático I ...,137,13,11,120,1,1,124,3,1,1
2,2025,5,Ingeniería en Sistemas de Información ...,2008,110,Ingeniería y Sociedad ...,22,2,2,17,0,0,20,1,2,2


## Silver

In [35]:
dsc_data_bronze_2008="Data/bronze/2008"
dsc_data_bronze_2023="Data/bronze/2023"
df_students_2008_bronze = DeltaTable(dsc_data_bronze_2008).to_pandas() 
df_students_2023_bronze = DeltaTable(dsc_data_bronze_2023).to_pandas() 

dsc_data_bronze_94 = "Data/bronze/1994"
dsc_data_bronze_95 = "Data/bronze/1995"
df_students_94_bronze = DeltaTable(dsc_data_bronze_94).to_pandas()
df_students_95_bronze = DeltaTable(dsc_data_bronze_95).to_pandas()

df_students_95_bronze.head(3)

 

,Año,especialid,esp,plan,materia,nombre,Total Inscriptos,Nuevos Inscriptos,Nuevos Inscriptos Libres,Recursantes Libres,Nuevos Inscriptos Regulares,Nuevos Inscritpos Promocionados,Recursantes,Recursantes Regulares,Recursantes Promocionados,__index_level_0__
0,2025,7,Ingeniería Eléctrica ...,95,101,Álgebra y Geometría Analítica ...,5,0,0,4,0,0,5,1,0,87
1,2025,7,Ingeniería Eléctrica ...,95,102,Análisis Matemático I ...,15,7,7,7,0,0,8,0,1,88
2,2025,7,Ingeniería Eléctrica ...,95,105,Física I ...,10,3,3,5,0,0,7,1,1,89


In [36]:
df_students_2008_bronze.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 16 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   Año                              60 non-null     int64 
 1   especialid                       60 non-null     int64 
 2   esp                              60 non-null     object
 3   plan                             60 non-null     int64 
 4   materia                          60 non-null     int64 
 5   nombre                           60 non-null     object
 6   Total Inscriptos                 60 non-null     int64 
 7   Nuevos Inscriptos                60 non-null     int64 
 8   Nuevos Inscriptos Libres         60 non-null     int64 
 9   Recursantes Libres               60 non-null     int64 
 10  Nuevos Inscriptos Regulares      60 non-null     int64 
 11  Nuevos Inscritpos Promocionados  60 non-null     int64 
 12  Recursantes                      60 no

In [37]:
dsc_data_bronze_2008 = "Data/bronze/2008"
dsc_data_bronze_2023 = "Data/bronze/2023"
dsc_data_bronze_94 = "Data/bronze/1994"
dsc_data_bronze_95 = "Data/bronze/1995"

df_students_2008_bronze = DeltaTable(dsc_data_bronze_2008).to_pandas()
df_students_2023_bronze = DeltaTable(dsc_data_bronze_2023).to_pandas()
df_students_94_bronze = DeltaTable(dsc_data_bronze_94).to_pandas()
df_students_95_bronze = DeltaTable(dsc_data_bronze_95).to_pandas()

df_students_95_bronze.head(3)


,Año,especialid,esp,plan,materia,nombre,Total Inscriptos,Nuevos Inscriptos,Nuevos Inscriptos Libres,Recursantes Libres,Nuevos Inscriptos Regulares,Nuevos Inscritpos Promocionados,Recursantes,Recursantes Regulares,Recursantes Promocionados,__index_level_0__
0,2025,7,Ingeniería Eléctrica ...,95,101,Álgebra y Geometría Analítica ...,5,0,0,4,0,0,5,1,0,87
1,2025,7,Ingeniería Eléctrica ...,95,102,Análisis Matemático I ...,15,7,7,7,0,0,8,0,1,88
2,2025,7,Ingeniería Eléctrica ...,95,105,Física I ...,10,3,3,5,0,0,7,1,1,89


In [38]:
df_students_2008_cleaned = df_students_2008_bronze
df_students_2023_cleaned = df_students_2023_bronze
df_students_94_cleaned = df_students_94_bronze
df_students_95_cleaned = df_students_95_bronze

renombres_columnas = {
    "Año": "Anio",
    "especialid": "Especialidad_ID",
    "esp": "Especialidad_Nombre",
    "plan": "Plan",
    "materia": "Materia_ID",
    "nombre": "Materia_Nombre",
    "Total Inscriptos": "Total_Inscriptos",
    "Nuevos Inscriptos": "Nuevos_Inscriptos",
    "Nuevos Inscriptos Libres": "Nuevos_Inscriptos_Libres",
    "Recursantes Libres": "Recursantes_Libres",
    "Nuevos Inscriptos Regulares": "Nuevos_Inscriptos_Regulares",
    "Nuevos Inscritpos Promocionados": "Nuevos_Inscriptos_Promocionados",
    "Recursantes": "Recursantes",
    "Recursantes Regulares": "Recursantes_Regulares",
    "Recursantes Promocionados": "Recursantes_Promocionados"
}

df_students_2008_cleaned = df_students_2008_cleaned.rename(columns=renombres_columnas)
df_students_2023_cleaned = df_students_2023_cleaned.rename(columns=renombres_columnas)
df_students_94_cleaned = df_students_94_cleaned.rename(columns=renombres_columnas)
df_students_95_cleaned = df_students_95_cleaned.rename(columns=renombres_columnas)
df_students_95_cleaned



,Anio,Especialidad_ID,Especialidad_Nombre,Plan,Materia_ID,Materia_Nombre,Total_Inscriptos,Nuevos_Inscriptos,Nuevos_Inscriptos_Libres,Recursantes_Libres,Nuevos_Inscriptos_Regulares,Nuevos_Inscriptos_Promocionados,Recursantes,Recursantes_Regulares,Recursantes_Promocionados,__index_level_0__
0,2025,7,Ingeniería Eléctrica ...,95,101,Álgebra y Geometría Analítica ...,5,0,0,4,0,0,5,1,0,87
1,2025,7,Ingeniería Eléctrica ...,95,102,Análisis Matemático I ...,15,7,7,7,0,0,8,0,1,88
2,2025,7,Ingeniería Eléctrica ...,95,105,Física I ...,10,3,3,5,0,0,7,1,1,89
3,2025,7,Ingeniería Eléctrica ...,95,107,Química General ...,4,1,1,3,0,0,3,0,0,90
4,2025,7,Ingeniería Eléctrica ...,95,108,Economía ...,30,20,12,9,7,1,10,1,0,91
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,2025,31,Ingeniería Civil ...,95,577,Puentes (Elec.) ...,45,19,9,0,10,0,26,26,0,338
133,2025,31,Ingeniería Civil ...,95,579,Gestión Ingenieril (Elec.) ...,42,17,9,0,0,8,25,0,25,339
134,2025,31,Ingeniería Civil ...,95,665,Inglés I ...,7,1,1,3,0,0,6,1,2,340
135,2025,31,Ingeniería Civil ...,95,666,Inglés II ...,28,17,10,8,0,7,11,2,1,341


In [39]:
for df in [df_students_2008_cleaned, df_students_2023_cleaned, df_students_94_cleaned, df_students_95_cleaned]:
    if "ID_Compuesto" not in df.columns:
        df["ID_Compuesto"] = (
            df["Especialidad_ID"].astype(str) + "_" +
            df["Plan"].astype(str) + "_" +
            df["Anio"].astype(str)
        )
    if "__index_level_0__" in df.columns:
        df.drop(columns=["__index_level_0__"], inplace=True)

df_students_95_cleaned


,Anio,Especialidad_ID,Especialidad_Nombre,Plan,Materia_ID,Materia_Nombre,Total_Inscriptos,Nuevos_Inscriptos,Nuevos_Inscriptos_Libres,Recursantes_Libres,Nuevos_Inscriptos_Regulares,Nuevos_Inscriptos_Promocionados,Recursantes,Recursantes_Regulares,Recursantes_Promocionados,ID_Compuesto
0,2025,7,Ingeniería Eléctrica ...,95,101,Álgebra y Geometría Analítica ...,5,0,0,4,0,0,5,1,0,7_95_2025
1,2025,7,Ingeniería Eléctrica ...,95,102,Análisis Matemático I ...,15,7,7,7,0,0,8,0,1,7_95_2025
2,2025,7,Ingeniería Eléctrica ...,95,105,Física I ...,10,3,3,5,0,0,7,1,1,7_95_2025
3,2025,7,Ingeniería Eléctrica ...,95,107,Química General ...,4,1,1,3,0,0,3,0,0,7_95_2025
4,2025,7,Ingeniería Eléctrica ...,95,108,Economía ...,30,20,12,9,7,1,10,1,0,7_95_2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,2025,31,Ingeniería Civil ...,95,577,Puentes (Elec.) ...,45,19,9,0,10,0,26,26,0,31_95_2025
133,2025,31,Ingeniería Civil ...,95,579,Gestión Ingenieril (Elec.) ...,42,17,9,0,0,8,25,0,25,31_95_2025
134,2025,31,Ingeniería Civil ...,95,665,Inglés I ...,7,1,1,3,0,0,6,1,2,31_95_2025
135,2025,31,Ingeniería Civil ...,95,666,Inglés II ...,28,17,10,8,0,7,11,2,1,31_95_2025


In [40]:
# Materias del Departamento de Ciencias Básicas
materias_ciencias_basicas = [
    "Análisis Matemático I",
    "Álgebra y Geometría Analítica",
    "Algebra y Geometría Analítica",
    "Análisis Matemático II",
    "Sistemas de Representación",
    "Fundamentos de la Informática",
    "Probabilidades y Estadísticas",
    "Física I",
    "Física II",
    "Química General",
    "Ingeniería y Sociedad",
    "Inglés I",
    "Inglés II",
    "Ingles I",
    "Ingles II",
    "Economía",
    "Legislación",
    "Ingeniería Legal",
]

# Normalizamos a minúsculas para comparar sin problemas de mayúsculas/tildes
materias_cb_lower = [m.lower().strip() for m in materias_ciencias_basicas]

for df in [df_students_2008_cleaned, df_students_2023_cleaned, df_students_94_cleaned, df_students_95_cleaned]:
    if "Ciencias_Basicas" not in df.columns:
        df["Ciencias_Basicas"] = df["Materia_Nombre"].str.lower().str.strip().isin(materias_cb_lower)

df_students_95_cleaned[["Materia_Nombre", "Ciencias_Basicas"]].drop_duplicates()


,Materia_Nombre,Ciencias_Basicas
0,Álgebra y Geometría Analítica ...,True
1,Análisis Matemático I ...,True
2,Física I ...,True
3,Química General ...,True
4,Economía ...,True
...,...,...
130,Análisis Estructural III (Elec.) ...,False
131,Prefabricación (Elec.) ...,False
132,Puentes (Elec.) ...,False
133,Gestión Ingenieril (Elec.) ...,False


In [41]:
for df in [df_students_2008_cleaned, df_students_2023_cleaned, df_students_94_cleaned, df_students_95_cleaned]:
    if "Total_Libres" not in df.columns:
        df["Total_Libres"] = df["Nuevos_Inscriptos_Libres"] + df["Recursantes_Libres"]
    if "Total_Regulares" not in df.columns:
        df["Total_Regulares"] = df["Nuevos_Inscriptos_Regulares"] + df["Recursantes_Regulares"]
    if "Total_Promocionados" not in df.columns:
        df["Total_Promocionados"] = df["Nuevos_Inscriptos_Promocionados"] + df["Recursantes_Promocionados"]

df_students_95_cleaned[df_students_95_cleaned["Especialidad_ID"] == 5]


,Anio,Especialidad_ID,Especialidad_Nombre,Plan,Materia_ID,Materia_Nombre,Total_Inscriptos,Nuevos_Inscriptos,Nuevos_Inscriptos_Libres,Recursantes_Libres,Nuevos_Inscriptos_Regulares,Nuevos_Inscriptos_Promocionados,Recursantes,Recursantes_Regulares,Recursantes_Promocionados,ID_Compuesto,Ciencias_Basicas,Total_Libres,Total_Regulares,Total_Promocionados


In [42]:
path_2008 = "Data/silver/2008"
path_2023 = "Data/silver/2023"
path_94 = "Data/silver/1994"
path_95 = "Data/silver/1995"

save_data_as_delta(df_students_2008_cleaned, path_2008, mode="overwrite")
save_data_as_delta(df_students_2023_cleaned, path_2023, mode="overwrite")
save_data_as_delta(df_students_94_cleaned, path_94, mode="overwrite")
save_data_as_delta(df_students_95_cleaned, path_95, mode="overwrite")


Datos almacenados en el Datalke en: Data/silver/2008
Datos almacenados en el Datalke en: Data/silver/2023
Datos almacenados en el Datalke en: Data/silver/1994
Datos almacenados en el Datalke en: Data/silver/1995


In [43]:
df_students_94_cleaned

,Anio,Especialidad_ID,Especialidad_Nombre,Plan,Materia_ID,Materia_Nombre,Total_Inscriptos,Nuevos_Inscriptos,Nuevos_Inscriptos_Libres,Recursantes_Libres,Nuevos_Inscriptos_Regulares,Nuevos_Inscriptos_Promocionados,Recursantes,Recursantes_Regulares,Recursantes_Promocionados,ID_Compuesto,Ciencias_Basicas,Total_Libres,Total_Regulares,Total_Promocionados
0,2025,17,Ingeniería Mecánica ...,94,101,Álgebra y Geometría Analítica ...,32,7,5,22,2,0,25,2,1,17_94_2025,True,27,4,1
1,2025,17,Ingeniería Mecánica ...,94,102,Análisis Matemático I ...,57,13,13,39,0,0,44,4,1,17_94_2025,True,52,4,1
2,2025,17,Ingeniería Mecánica ...,94,105,Física I ...,15,5,3,10,2,0,10,0,0,17_94_2025,True,13,2,0
3,2025,17,Ingeniería Mecánica ...,94,107,Química General ...,13,2,2,10,0,0,11,1,0,17_94_2025,True,12,1,0
4,2025,17,Ingeniería Mecánica ...,94,110,Ingeniería y Sociedad ...,7,0,0,6,0,0,7,0,1,17_94_2025,True,6,0,1
5,2025,17,Ingeniería Mecánica ...,94,112,Fundamentos de Informática ...,25,1,1,22,0,0,24,1,1,17_94_2025,False,23,1,1
6,2025,17,Ingeniería Mecánica ...,94,115,Sistemas de Representación ...,15,0,0,13,0,0,15,0,2,17_94_2025,True,13,0,2
7,2025,17,Ingeniería Mecánica ...,94,121,Ingeniería Mecánica I ...,10,0,0,8,0,0,10,1,1,17_94_2025,False,8,1,1
8,2025,17,Ingeniería Mecánica ...,94,124,Ingeniería Ambiental y Seguridad Industrial ...,16,8,3,4,0,5,8,0,4,17_94_2025,False,7,0,9
9,2025,17,Ingeniería Mecánica ...,94,172,Computación Aplicada (Elec.) ...,27,23,2,2,4,17,4,1,1,17_94_2025,False,4,5,18
